In [1]:
import pathlib

orig_instance_dir = pathlib.Path("original_instances")
original_instances = list(orig_instance_dir.rglob("*.txt"))

In [2]:
def read_orig_instance(instance_path: pathlib.Path)->str:
    with open(instance_path.absolute(), "r") as f:
        return f.read()

In [37]:
import re
from typing import Tuple

def read_customers(text: str)->list[str]:
    lines = text.split(";")
    customers = []
    for line in lines:
        pattern = r"set\sV\s*:="
        if re.search(pattern, line):
            pattern = r"(C\d+)"
            customers = re.findall(pattern, line)
            break
    customers = sorted(customers, key=lambda c: int(c[1:]))
    return customers

def read_stations(text: str)->list[str]:
    lines = text.split(";")
    for line in lines:
        pattern = r"set\sF\s*:="
        if re.search(pattern, line):
            pattern = r"(F\S+)"
            return re.findall(pattern, line)
    return []

def generate_node_mapping(text:str)->Tuple[dict[str, int], dict[str, list[str]]]:
    node_map:dict[str, int] = {}
    customers = read_customers(text)
    stations = read_stations(text)
    uniq_stations = [station for station in stations if "_" not in station]
    num_nodes = len(customers + stations)+2
    node_ids = ["N0"] + customers
    station_groups_by_uniq_station: dict[str, list[str]] = {}
    for uniq_station in uniq_stations:
        node_ids += [uniq_station]
        station_groups_by_uniq_station[uniq_station] = [uniq_station]
        for station in stations:
            if uniq_station+"_" in station:
                node_ids += [station]
                station_groups_by_uniq_station[uniq_station] += [station]
    node_ids += ["Nσ"]
    for i, node_id in enumerate(node_ids):
        node_map[node_id] = i
    return node_map, station_groups_by_uniq_station
        
def convert_instance(text: str) -> Tuple[str, str]:
    node_map, station_groups_by_uniq_station = generate_node_mapping(text)
    print(node_map)
    new_text = text
    for node_id, node_idx in node_map.items():
        new_text = new_text.replace(node_id+" ", str(node_idx)+" ")
    new_text = new_text.replace("N0", str(node_map["N0"]))
    new_text = new_text.replace("Nσ", str(node_map["Nσ"]))
    new_text = new_text.replace("σ", "sig")
    uniq_text = "F_UNIQ :="
    station_groups_text = ""
    for uniq, station_group in station_groups_by_uniq_station.items():
        uniq_text += f" {node_map[uniq]}"
        station_group_txt = f"F_NODES[{node_map[uniq]}] :="
        for station in station_group:
            station_group_txt += f" {node_map[station]}"
        station_group_txt += ";\n"
        station_groups_text += station_group_txt
    uniq_text += ";\n"
    new_text = uniq_text + station_groups_text + new_text
    return new_text, str(node_map)

In [39]:
for instance_path in original_instances:
    new_instance_dir = pathlib.Path("new_instances")
    instance_set_dir = instance_path.parent.stem
    instance_name = instance_path.name
    instance_stem = instance_path.stem
    new_instance_path = new_instance_dir/instance_set_dir/instance_name
    mapping_path = new_instance_dir/instance_set_dir/f"{instance_stem}.map"
    new_instance_path.parent.mkdir(parents=True, exist_ok=True)
    mapping_path.parent.mkdir(parents=True, exist_ok=True)
    
    old_instance_text = read_orig_instance(instance_path)
    new_text, node_map_text = convert_instance(old_instance_text)
    mapping_path.write_text(node_map_text)
    new_instance_path.write_text(new_text)

{'N0': 0, 'C1': 1, 'C2': 2, 'C3': 3, 'C4': 4, 'C5': 5, 'C6': 6, 'C7': 7, 'C8': 8, 'C9': 9, 'C10': 10, 'F11': 11, 'F11_1': 12, 'F11_2': 13, 'F11_3': 14, 'F11_4': 15, 'F12': 16, 'F12_1': 17, 'F12_2': 18, 'F12_3': 19, 'F12_4': 20, 'F13': 21, 'F13_1': 22, 'F13_2': 23, 'F13_3': 24, 'F13_4': 25, 'F14': 26, 'F14_1': 27, 'F14_2': 28, 'F14_3': 29, 'F14_4': 30, 'F15': 31, 'F15_1': 32, 'F15_2': 33, 'F15_3': 34, 'F15_4': 35, 'Nσ': 36}
{'N0': 0, 'C1': 1, 'C2': 2, 'C3': 3, 'C4': 4, 'C5': 5, 'C6': 6, 'C7': 7, 'C8': 8, 'C9': 9, 'C10': 10, 'F11': 11, 'F11_1': 12, 'F11_2': 13, 'F11_3': 14, 'F11_4': 15, 'F12': 16, 'F12_1': 17, 'F12_2': 18, 'F12_3': 19, 'F12_4': 20, 'F13': 21, 'F13_1': 22, 'F13_2': 23, 'F13_3': 24, 'F13_4': 25, 'F14': 26, 'F14_1': 27, 'F14_2': 28, 'F14_3': 29, 'F14_4': 30, 'Nσ': 31}
{'N0': 0, 'C1': 1, 'C2': 2, 'C3': 3, 'C4': 4, 'C5': 5, 'C6': 6, 'C7': 7, 'C8': 8, 'C9': 9, 'C10': 10, 'F11': 11, 'F11_1': 12, 'F11_2': 13, 'F11_3': 14, 'F11_4': 15, 'F12': 16, 'F12_1': 17, 'F12_2': 18, 'F12_3'